# 🧠 Deflationary Mamba Extractor (DME) V2
## 5-Speaker Speech Separation on Libri5Mix

**Architecture:** Iterative deflationary extraction with dual-path bidirectional Mamba SSM separator  
**Key Innovation:** No Permutation Invariant Training (PIT) needed — greedy sequential extraction with learned stopping  
**Dataset:** Libri5Mix (5000 train / 1000 dev / 1000 test @ 16 kHz)  


---


In [ ]:
# ── Cell 1.1: Install Dependencies (offline from local Kaggle datasets) ──
# Note: numpy and scipy are intentionally skipped because Kaggle already has them pre-installed,
# and upgrading to numpy 2.x breaks pre-installed packages (like numba, ydata-profiling).
!pip install -q /kaggle/input/datasets/faaizhussain/pip-wheels/pip-wheels/causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl --no-deps
!pip install -q /kaggle/input/datasets/faaizhussain/pip-wheels/pip-wheels/mamba_ssm-2.3.2.post1-cp312-cp312-linux_x86_64.whl --no-deps
!pip install -q /kaggle/input/datasets/faaizhussain/pip-wheels/pip-wheels/pesq-0.0.4-cp312-cp312-linux_x86_64.whl --no-deps
!pip install -q /kaggle/input/datasets/faaizhussain/pip-wheels/pip-wheels/pystoi-0.4.1-py2.py3-none-any.whl --no-deps
print("All dependencies installed!")


In [ ]:
# ── Cell 1.2: Imports ──
import os, sys, math, random, time, warnings, gc
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

from pesq import pesq as pesq_fn
from pystoi import stoi as stoi_fn

try:
    from mamba_ssm import Mamba
    print("mamba_ssm imported")
except ImportError:
    from mamba_ssm.modules.mamba_simple import Mamba
    print("mamba_ssm.modules.mamba_simple imported")

try:
    from transformers import WavLMModel
    WAVLM_AVAILABLE = True
    print("WavLM available")
except ImportError:
    WAVLM_AVAILABLE = False
    print("WavLM not available - will use encoder-only features")

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (12, 6),
    "font.family": "sans-serif",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})
PALETTE = sns.color_palette("husl", 8)
print("All imports successful!")


In [ ]:
# ── Cell 1.3: Configuration ──
@dataclass
class DMEConfig:
    # ── Data ──
    data_root: str = "/kaggle/input/datasets/mruddunijmodha/librimix-s5/Libri5Mix"
    sample_rate: int = 16000
    segment_len: float = 3.0          # seconds per training clip
    num_speakers: int = 5

    # ── Encoder / Decoder ──
    enc_kernel: int = 16
    enc_stride: int = 8
    enc_dim: int = 256

    # ── Feature fusion ──
    feature_dim: int = 256
    use_wavlm: bool = True
    wavlm_path: str = "/kaggle/input/wavlm-base/wavlm-base"

    # ── Separator ──
    num_mamba_blocks: int = 4
    mamba_d_state: int = 16
    mamba_d_conv: int = 4
    mamba_expand: int = 2
    chunk_size: int = 100             # dual-path chunk length

    # ── Extraction ──
    max_iterations: int = 6           # N_speakers + 1 termination probe
    confidence_threshold: float = 0.5

    # ── Refinement ──
    use_refinement: bool = True
    refine_layers: int = 2

    # ── Training ──
    batch_size: int = 2               # Reduced to 2 for maximum safety against OOM
    grad_accum_steps: int = 8         # Effective batch size = 16
    lr: float = 1.5e-4
    weight_decay: float = 1e-2
    num_epochs: int = 50
    grad_clip: float = 5.0
    warmup_steps: int = 500

    # ── Checkpoint Resume ──
    resume_ckpt: str = ""             # Set to checkpoint path to resume, e.g.:
    # "/kaggle/input/dme-checkpoint/dme_ckpt_latest.pth"

    # ── Loss weights ──
    alpha_confidence: float = 0.1
    alpha_consistency: float = 0.5    # v2: increased from 0.1 to fix SDRi
    alpha_l1: float = 0.3             # v2: NEW L1 waveform loss

    # ── Output ──
    save_dir: str = "/kaggle/working"

config = DMEConfig()

# Override WavLM if not available
if not WAVLM_AVAILABLE:
    config.use_wavlm = False
if config.use_wavlm and not os.path.exists(config.wavlm_path):
    print(f"WavLM path not found: {config.wavlm_path} -- disabling WavLM")
    config.use_wavlm = False

print(f"Config ready | WavLM={config.use_wavlm} | epochs={config.num_epochs} | batch={config.batch_size}")


In [ ]:
# ── Cell 1.4: Device & Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM   : {mem_gb:.1f} GB")

os.makedirs(f"{config.save_dir}/figures", exist_ok=True)


---
## 📂 Section 2: Dataset & DataLoader


In [ ]:
# ── Cell 2.1: Libri5Mix Dataset Class ──
class Libri5MixDataset(Dataset):
    """Loads Libri5Mix mixtures and their 5 constituent sources."""

    SRC_COLS = ["s1_path", "s2_path", "s3_path", "s4_path", "s5_path"]

    def __init__(self, csv_path, data_root, segment_len=None,
                 sample_rate=16000, random_crop=True):
        self.data_root = data_root
        self.segment_len = segment_len
        self.sr = sample_rate
        self.random_crop = random_crop
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def _remap(self, path):
        return path.replace("/kaggle/working/Libri5Mix", self.data_root)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mix, _ = torchaudio.load(self._remap(row["mix_path"]))   # (1, L)
        srcs = [torchaudio.load(self._remap(row[c]))[0] for c in self.SRC_COLS]

        L = mix.shape[-1]
        if self.segment_len is not None:
            S = int(self.segment_len * self.sr)
            if L > S:
                start = random.randint(0, L - S) if self.random_crop else 0
                mix = mix[:, start:start + S]
                srcs = [s[:, start:start + S] for s in srcs]
            elif L < S:
                mix = F.pad(mix, (0, S - L))
                srcs = [F.pad(s, (0, S - L)) for s in srcs]
        return mix, srcs

def collate_fn(batch):
    mixtures = torch.stack([b[0] for b in batch])
    sources = [torch.stack([b[1][i] for b in batch]) for i in range(5)]
    return mixtures, sources


In [ ]:
# ── Cell 2.2: Create DataLoaders ──
csv_dir = os.path.join(config.data_root, "metadata")

train_ds = Libri5MixDataset(
    os.path.join(csv_dir, "libri5mix_train-100.csv"),
    config.data_root, segment_len=config.segment_len,
    sample_rate=config.sample_rate, random_crop=True,
)
val_ds = Libri5MixDataset(
    os.path.join(csv_dir, "libri5mix_dev.csv"),
    config.data_root, segment_len=config.segment_len,
    sample_rate=config.sample_rate, random_crop=False,
)
test_ds = Libri5MixDataset(
    os.path.join(csv_dir, "libri5mix_test.csv"),
    config.data_root, segment_len=None,             # full length
    sample_rate=config.sample_rate, random_crop=False,
)

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                          num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Steps/epoch: {len(train_loader)}")


In [ ]:
# ── Cell 2.3: Dataset Visualisation — Graphs 1-3 ──

# ---- Graph 1: Histogram of audio durations ----
durations = {"train": [], "dev": [], "test": []}
for split, ds in [("train", train_ds), ("dev", val_ds), ("test", test_ds)]:
    for idx in range(len(ds)):
        row = ds.df.iloc[idx]
        waveform, sr = torchaudio.load(ds._remap(row["mix_path"]))
        durations[split].append(waveform.shape[-1] / sr)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (split, durs) in enumerate(durations.items()):
    ax.hist(durs, bins=30, alpha=0.6, label=f"{split} (n={len(durs)})", color=PALETTE[i])
ax.set_xlabel("Duration (s)")
ax.set_ylabel("Count")
ax.set_title("Graph 1 — Audio Duration Distribution")
ax.legend()
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph01_durations.png", dpi=150)
plt.show()

# ---- Graph 2: Waveform of sample mixture + sources ----
sample_mix, sample_srcs = train_ds[0]
fig, axes = plt.subplots(6, 1, figsize=(14, 10), sharex=True)
t = np.arange(sample_mix.shape[-1]) / config.sample_rate
axes[0].plot(t, sample_mix[0].numpy(), color=PALETTE[0], linewidth=0.4)
axes[0].set_title("Mixture")
axes[0].set_ylabel("Amplitude")
for i, src in enumerate(sample_srcs):
    axes[i + 1].plot(t, src[0].numpy(), color=PALETTE[i + 1], linewidth=0.4)
    axes[i + 1].set_title(f"Source {i + 1}")
    axes[i + 1].set_ylabel("Amplitude")
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Graph 2 — Sample Mixture & Source Waveforms", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph02_waveforms.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Graph 3: Spectrograms ----
fig, axes = plt.subplots(6, 1, figsize=(14, 12), sharex=True)
spec_transform = torchaudio.transforms.Spectrogram(n_fft=512, hop_length=128, power=2.0)
for ax_i, (sig, title) in enumerate(
    [(sample_mix, "Mixture")] + [(s, f"Source {i+1}") for i, s in enumerate(sample_srcs)]
):
    spec = spec_transform(sig)[0]                         # (F, T)
    spec_db = 10 * torch.log10(spec.clamp(min=1e-10))
    axes[ax_i].imshow(spec_db.numpy(), origin="lower", aspect="auto",
                      cmap="magma", extent=[0, sig.shape[-1]/config.sample_rate, 0, config.sample_rate/2])
    axes[ax_i].set_title(title)
    axes[ax_i].set_ylabel("Hz")
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Graph 3 — Spectrograms", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph03_spectrograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dataset visualisation complete!")


---
## 🏗️ Section 3: Model Architecture

**Deflationary Mamba Extractor (DME)**

```
Mixture ─► Encoder ─► Feature Fusion ─┐
                                       ▼
                              ┌── Iteration Loop ──┐
                              │  Dual-Path Mamba   │
                              │  Mask Head → mask  │
                              │  Conf Head → conf  │
                              │  speaker = mask⊙res│
                              │  res = res - speaker│
                              └────────────────────┘
                                       ▼
                              Refinement (opt.)
                                       ▼
                              Decoder ─► Waveforms
```


In [ ]:
# ── Cell 3.1: Encoder & Decoder ──

class Encoder(nn.Module):
    def __init__(self, out_channels, kernel_size, stride):
        super().__init__()
        self.conv = nn.Conv1d(1, out_channels, kernel_size, stride=stride, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.conv(x))          # (B,1,L) -> (B,N,T)


class Decoder(nn.Module):
    def __init__(self, in_channels, kernel_size, stride):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(in_channels, 1, kernel_size, stride=stride, bias=False)

    def forward(self, x):
        return self.deconv(x)                    # (B,N,T) -> (B,1,L')

print("Encoder / Decoder defined")


In [ ]:
# ── Cell 3.2: Bidirectional Mamba & Dual-Path Mamba Block ──

class BidirectionalMamba(nn.Module):
    """Run Mamba in both directions and sum outputs."""
    def __init__(self, d_model, d_state, d_conv, expand):
        super().__init__()
        self.fwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.bwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x):
        return self.fwd(x) + self.bwd(x.flip(1)).flip(1)


class DualPathMambaBlock(nn.Module):
    """
    Intra-chunk: local patterns within each chunk (BidirMamba).
    Inter-chunk: global patterns across chunks  (BidirMamba).
    Non-overlapping chunking with padding.
    """
    def __init__(self, d_model, d_state, d_conv, expand, chunk_size):
        super().__init__()
        self.K = chunk_size
        # intra
        self.norm1  = nn.LayerNorm(d_model)
        self.intra  = BidirectionalMamba(d_model, d_state, d_conv, expand)
        self.fc1    = nn.Linear(d_model, d_model)
        # inter
        self.norm2  = nn.LayerNorm(d_model)
        self.inter  = BidirectionalMamba(d_model, d_state, d_conv, expand)
        self.fc2    = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        # pad to multiple of chunk_size
        rest = (self.K - T % self.K) % self.K
        if rest:
            x = F.pad(x, (0, 0, 0, rest))
        Tp = x.shape[1]
        S = Tp // self.K                               # num chunks

        x = x.reshape(B, S, self.K, D)

        # ── intra-chunk ──
        res = x
        z = self.norm1(x).reshape(B * S, self.K, D)
        z = self.fc1(self.intra(z)).reshape(B, S, self.K, D)
        x = res + z

        # ── inter-chunk ──
        res = x
        z = x.permute(0, 2, 1, 3).reshape(B * self.K, S, D)
        z = self.norm2(z)
        z = self.fc2(self.inter(z)).reshape(B, self.K, S, D).permute(0, 2, 1, 3)
        x = res + z

        x = x.reshape(B, Tp, D)
        if rest:
            x = x[:, :T, :]
        return x

print("Dual-Path Mamba defined")


In [ ]:
# ── Cell 3.3: WavLM Feature Extractor (optional) ──

class WavLMFeatures(nn.Module):
    """Frozen WavLM-base producing 768-dim features at 50 Hz."""
    def __init__(self, model_path):
        super().__init__()
        self.model = WavLMModel.from_pretrained(model_path)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self, wav):
        # wav: (B, L) raw 16 kHz waveform
        return self.model(wav).last_hidden_state       # (B, T', 768)

if config.use_wavlm:
    print("WavLM feature extractor defined")
else:
    print("WavLM disabled -- skipping")


In [ ]:
# ── Cell 3.4: Refinement Module ──

class RefinementModule(nn.Module):
    """Shared lightweight Mamba applied independently to each extracted speaker."""
    def __init__(self, d_model, d_state, d_conv, expand, num_layers):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms  = nn.ModuleList()
        for _ in range(num_layers):
            self.norms.append(nn.LayerNorm(d_model))
            self.layers.append(BidirectionalMamba(d_model, d_state, d_conv, expand))

    def forward(self, feats_list):
        """feats_list: list of (B, N, T) tensors."""
        refined = []
        for feat in feats_list:
            x = feat.transpose(1, 2)                    # (B, T, N)
            for norm, layer in zip(self.norms, self.layers):
                x = x + layer(norm(x))
            refined.append(x.transpose(1, 2))            # (B, N, T)
        return refined

print("Refinement module defined")


In [ ]:
# ── Cell 3.5: Full DME Model ──

class DeflatMambaExtractor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # encoder / decoder
        self.encoder = Encoder(cfg.enc_dim, cfg.enc_kernel, cfg.enc_stride)
        self.decoder = Decoder(cfg.enc_dim, cfg.enc_kernel, cfg.enc_stride)

        # wavlm (optional)
        self.use_wavlm = cfg.use_wavlm
        if self.use_wavlm:
            self.wavlm = WavLMFeatures(cfg.wavlm_path)
            in_dim = cfg.enc_dim + 768
        else:
            in_dim = cfg.enc_dim

        # feature projection
        self.feat_proj = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, cfg.feature_dim))

        # shared separator (dual-path mamba)
        self.separator = nn.ModuleList([
            DualPathMambaBlock(cfg.feature_dim, cfg.mamba_d_state, cfg.mamba_d_conv,
                               cfg.mamba_expand, cfg.chunk_size)
            for _ in range(cfg.num_mamba_blocks)
        ])

        # heads
        self.mask_head = nn.Sequential(nn.Linear(cfg.feature_dim, cfg.enc_dim), nn.ReLU())
        self.conf_head = nn.Sequential(nn.Linear(cfg.feature_dim, 1), nn.Sigmoid())

        # refinement (optional)
        if cfg.use_refinement:
            self.refinement = RefinementModule(
                cfg.enc_dim, cfg.mamba_d_state, cfg.mamba_d_conv, cfg.mamba_expand, cfg.refine_layers
            )

    # ── forward ──
    def forward(self, mixture):
        B, _, L_orig = mixture.shape

        # pad so encoder output is clean
        rem = (L_orig - self.cfg.enc_kernel) % self.cfg.enc_stride
        if rem:
            mixture = F.pad(mixture, (0, self.cfg.enc_stride - rem))

        enc = self.encoder(mixture)                      # (B, N, T)

        # WavLM features (optional)
        wavlm_feat = None
        if self.use_wavlm:
            wav_in = mixture.squeeze(1)                  # (B, L)
            wavlm_feat = self.wavlm(wav_in)              # (B, T', 768)
            wavlm_feat = F.interpolate(
                wavlm_feat.transpose(1, 2), size=enc.shape[2],
                mode="linear", align_corners=False
            ).transpose(1, 2)                            # (B, T, 768)

        # iterative extraction
        enc_res = enc                                    # (B, N, T)
        speakers, confs = [], []

        for _ in range(self.cfg.max_iterations):
            # project residual to feature space
            feat = enc_res.transpose(1, 2)               # (B, T, N)
            if self.use_wavlm and wavlm_feat is not None:
                feat = torch.cat([feat, wavlm_feat], dim=-1)
            feat = self.feat_proj(feat)                  # (B, T, D)

            # separator
            for blk in self.separator:
                feat = blk(feat)

            # mask & confidence
            mask = self.mask_head(feat).transpose(1, 2)  # (B, N, T)
            conf = self.conf_head(feat.mean(dim=1)).squeeze(-1)  # (B,)

            # extract (v2: gradient-stop prevents error compounding)
            spk_enc = mask * enc_res
            enc_res = enc_res - spk_enc.detach()

            speakers.append(spk_enc)
            confs.append(conf)

            # early stop at inference
            if not self.training and conf.mean().item() < self.cfg.confidence_threshold:
                break

        # refinement
        if self.cfg.use_refinement and hasattr(self, "refinement"):
            speakers = self.refinement(speakers)

        # decode
        estimates = [self.decoder(s)[:, :, :L_orig] for s in speakers]
        return estimates, confs

print("DME model class defined")


In [ ]:
# ── Cell 3.6: Instantiate & Summarise Model ──
model = DeflatMambaExtractor(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")

# quick shape sanity check
_dummy = torch.randn(2, 1, int(config.sample_rate * config.segment_len), device=device)
with torch.no_grad():
    _ests, _confs = model(_dummy)
print(f"Input shape  : {_dummy.shape}")
print(f"# estimates  : {len(_ests)}, each {_ests[0].shape}")
print(f"# confidences: {len(_confs)}, each {_confs[0].shape}")
del _dummy, _ests, _confs
torch.cuda.empty_cache()
print("Shape sanity check passed!")


In [ ]:
# ── Cell 3.7: Graph 4 — Architecture Diagram ──

fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(0, 10); ax.set_ylim(0, 14)
ax.axis("off")

def box(x, y, w, h, text, color="#4a90d9", fontsize=9):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="white",
                                linewidth=1.5, alpha=0.85, zorder=2))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center",
            fontsize=fontsize, color="white", fontweight="bold", zorder=3)

def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=1.5, color="grey"))

# draw
box(3.5, 12.5, 3, 0.8, "Mixture  x(t)", "#2c3e50", 11)
arrow(5, 12.5, 5, 12.0)

box(1, 11, 3, 0.8, "1-D Conv Encoder", "#3498db")
box(6, 11, 3, 0.8, "WavLM (frozen)", "#e67e22")
arrow(5, 12.5, 2.5, 11.8)
arrow(5, 12.5, 7.5, 11.8)
arrow(2.5, 11, 5, 10.5)
arrow(7.5, 11, 5, 10.5)

box(3, 9.5, 4, 0.8, "Feature Fusion  (concat + proj)", "#1abc9c")
arrow(5, 9.5, 5, 9.0)

box(2, 6.8, 6, 2.0, "", "#8e44ad")
ax.text(5, 8.4, "Iteration Loop  (i = 1 .. 6)", ha="center", va="center",
        fontsize=10, color="white", fontweight="bold")
box(2.3, 7.0, 2.5, 0.6, "Dual-Path Mamba\n(Intra + Inter)", "#9b59b6", 8)
box(5.2, 7.5, 2.5, 0.5, "Mask Head", "#e74c3c", 8)
box(5.2, 7.0, 2.5, 0.4, "Conf Head", "#e74c3c", 8)

arrow(5, 6.8, 5, 6.2)
box(3, 5.5, 4, 0.6, "Refinement (2-layer Mamba)", "#27ae60")
arrow(5, 5.5, 5, 5.0)
box(3.5, 4.2, 3, 0.6, "Decoder  -> Waveforms", "#2c3e50")

ax.set_title("Graph 4 — DME Architecture", fontsize=14, pad=10)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph04_architecture.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 📉 Section 4: Loss Functions

1. **Deflationary Extraction Loss** — greedy 1-to-1 matching, no PIT  
2. **Confidence Loss** — BCE-style for speaker-present classification  
3. **Mixture Consistency Loss** — ‖Σ estimates − mixture‖²


In [ ]:
# ── Cell 4.1: SI-SNR (Scale-Invariant Signal-to-Noise Ratio) ──

def si_snr(estimate, target):
    """
    Args: estimate, target — shape (..., L)
    Returns: SI-SNR in dB, shape (...)
    """
    estimate = estimate - estimate.mean(dim=-1, keepdim=True)
    target   = target   - target.mean(dim=-1, keepdim=True)

    dot    = (estimate * target).sum(dim=-1, keepdim=True)
    energy = (target * target).sum(dim=-1, keepdim=True).clamp(min=1e-8)
    proj   = dot * target / energy

    noise  = estimate - proj
    snr    = 10 * torch.log10(
        (proj ** 2).sum(dim=-1).clamp(min=1e-8) /
        (noise ** 2).sum(dim=-1).clamp(min=1e-8)
    )
    return snr


def si_snr_np(estimate, target):
    """Numpy version for evaluation."""
    estimate = estimate - np.mean(estimate)
    target   = target   - np.mean(target)
    dot    = np.sum(estimate * target)
    energy = np.sum(target ** 2) + 1e-8
    proj   = dot * target / energy
    noise  = estimate - proj
    return float(10 * np.log10((np.sum(proj**2) + 1e-8) / (np.sum(noise**2) + 1e-8)))


def sdr_np(estimate, target):
    """Signal-to-Distortion Ratio (dB)."""
    noise = estimate - target
    return float(10 * np.log10((np.sum(target**2) + 1e-8) / (np.sum(noise**2) + 1e-8)))


print("SI-SNR / SDR functions defined")


In [ ]:
# ── Cell 4.2: Deflationary Extraction Loss ──

def deflationary_loss(estimates, targets, confidences, mixture, cfg):
    """
    Greedy 1-to-1 matching loss — no PIT permutation search.
    Returns: total_loss, loss_dict, per_iteration_snrs
    """
    K = len(estimates)        # max_iterations (6)
    N = len(targets)          # num_speakers  (5)
    B = estimates[0].shape[0]

    # flatten channel dim: (B,1,L) -> (B,L)
    est_f = [e.squeeze(1) for e in estimates]
    tgt_f = [t.squeeze(1) for t in targets]

    # pairwise SI-SNR: (K, N, B)
    pw = torch.zeros(K, N, B, device=estimates[0].device)
    for i in range(K):
        for j in range(N):
            pw[i, j] = si_snr(est_f[i], tgt_f[j])

    sep_loss  = torch.zeros(B, device=estimates[0].device)
    conf_loss = torch.zeros(B, device=estimates[0].device)
    per_iter  = [0.0] * K          # avg SI-SNR per iteration (for logging)
    conf_correct = 0
    conf_total   = 0

    for b in range(B):
        remaining = list(range(N))
        for i in range(K):
            c = confidences[i][b]
            if not remaining:
                # termination iteration — no speaker left
                conf_loss[b] += -torch.log(1 - c + 1e-8)
                conf_correct += int(c.item() < 0.5)
            else:
                scores   = pw[i, remaining, b]
                best_loc = scores.argmax().item()
                best_g   = remaining[best_loc]

                sep_loss[b]  += -pw[i, best_g, b]
                conf_loss[b] += -torch.log(c + 1e-8)
                per_iter[i]  += pw[i, best_g, b].item()
                conf_correct += int(c.item() >= 0.5)
                remaining.remove(best_g)
            conf_total += 1

    sep_loss  = sep_loss.mean()
    conf_loss = conf_loss.mean()

    # mixture consistency
    est_sum = sum(estimates)
    consist = F.mse_loss(est_sum, mixture)

    # v2: L1 waveform loss — penalizes amplitude errors directly
    l1_loss = torch.tensor(0.0, device=estimates[0].device)
    for b in range(B):
        remaining_l1 = list(range(N))
        for i in range(K):
            if not remaining_l1:
                break
            scores_l1 = pw[i, remaining_l1, b]
            best_loc_l1 = scores_l1.argmax().item()
            best_g_l1 = remaining_l1[best_loc_l1]
            l1_loss = l1_loss + F.l1_loss(est_f[i], tgt_f[best_g_l1])
            remaining_l1.remove(best_g_l1)
    l1_loss = l1_loss / max(B * min(K, N), 1)

    total = (sep_loss
             + cfg.alpha_confidence * conf_loss
             + cfg.alpha_consistency * consist
             + cfg.alpha_l1 * l1_loss)

    info = {
        "sep":   sep_loss.item(),
        "conf":  conf_loss.item(),
        "consist": consist.item(),
        "l1":    l1_loss.item(),
        "total": total.item(),
        "conf_acc": conf_correct / max(conf_total, 1),
    }
    per_iter = [v / max(B, 1) for v in per_iter]
    return total, info, per_iter

print("Deflationary loss defined")


---
## 🏋️ Section 5: Training


In [ ]:
# ── Cell 5.1: Training Utilities ──

def get_lr(step, warmup, peak_lr, total_steps):
    if step < warmup:
        return peak_lr * step / max(warmup, 1)
    progress = (step - warmup) / max(total_steps - warmup, 1)
    return peak_lr * 0.5 * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def quick_val(model, loader, cfg):
    """Fast validation: average SI-SNRi on dev set."""
    model.eval()
    snri_vals = []
    for mix, srcs in loader:
        mix  = mix.to(device)
        srcs = [s.to(device) for s in srcs]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            ests, _ = model(mix)
        
        # Convert back to float32 for metric calculation
        ests = [e.float() for e in ests]
        
        K = len(ests)
        N = len(srcs)
        B = mix.shape[0]

        est_f = [e.squeeze(1) for e in ests]
        tgt_f = [s.squeeze(1) for s in srcs]
        mix_f = mix.squeeze(1)

        pw = torch.zeros(K, N, B, device=device)
        for i in range(K):
            for j in range(N):
                pw[i, j] = si_snr(est_f[i], tgt_f[j])

        for b in range(B):
            remaining = list(range(N))
            for i in range(K):
                if not remaining:
                    break
                scores = pw[i, remaining, b]
                best_loc = scores.argmax().item()
                best_g = remaining[best_loc]

                snr_est = pw[i, best_g, b].item()
                snr_mix = si_snr(mix_f[b:b+1], tgt_f[best_g][b:b+1]).item()
                snri_vals.append(snr_est - snr_mix)
                remaining.remove(best_g)

    model.train()
    return np.mean(snri_vals) if snri_vals else 0.0

print("Training utilities ready")


In [ ]:
# ── Cell 5.2: Train DME ──

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.lr, weight_decay=config.weight_decay,
)

total_steps = config.num_epochs * (len(train_loader) // config.grad_accum_steps)
global_step = 0
best_val_snri = -1e9
start_epoch = 1

# history for graphs
history = {
    "train_loss": [],
    "val_sisnri": [],
    "conf_acc": [],
    "per_iter_snr": [[] for _ in range(config.max_iterations)],
    "grad_norms": defaultdict(list),
}

# ── Checkpoint Resume ──
if config.resume_ckpt and os.path.exists(config.resume_ckpt):
    ckpt = torch.load(config.resume_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["opt"])

    # merge history from previous session
    prev_history = ckpt["history"]
    for key in ["train_loss", "val_sisnri", "conf_acc"]:
        history[key] = prev_history[key]
    history["per_iter_snr"] = prev_history["per_iter_snr"]
    if isinstance(prev_history["grad_norms"], dict):
        history["grad_norms"] = defaultdict(list, prev_history["grad_norms"])

    start_epoch = ckpt["epoch"] + 1
    best_val_snri = max(history["val_sisnri"]) if history["val_sisnri"] else -1e9
    print(f"Resumed from epoch {ckpt['epoch']}, starting at epoch {start_epoch}")
    print(f"Previous best val SI-SNRi: {best_val_snri:.2f} dB")
    del ckpt
    torch.cuda.empty_cache()
else:
    print("Starting fresh training (no checkpoint found)")

end_epoch = start_epoch + config.num_epochs
print(f"Training: epochs {start_epoch}..{end_epoch - 1}, {total_steps} optimizer steps")
t0 = time.time()

for epoch in range(start_epoch, end_epoch):
    model.train()
    optimizer.zero_grad()
    ep_loss, ep_conf_acc = 0.0, 0.0
    ep_iter_snr = [0.0] * config.max_iterations
    ep_grad = defaultdict(float)
    n_batches = 0

    for mix, srcs in tqdm(train_loader, desc=f"Epoch {epoch}/{end_epoch - 1}", leave=False):
        mix  = mix.to(device)
        srcs = [s.to(device) for s in srcs]

        # lr schedule
        lr = get_lr(global_step, config.warmup_steps, config.lr, total_steps)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            ests, confs = model(mix)
            loss, info, piter = deflationary_loss(ests, srcs, confs, mix, config)
        
        # Scale loss for gradient accumulation
        loss = loss / config.grad_accum_steps
        loss.backward()

        # Step optimizer only after grad_accum_steps batches
        if (n_batches + 1) % config.grad_accum_steps == 0 or (n_batches + 1) == len(train_loader):
            # gradient norms per component
            for name, p in model.named_parameters():
                if p.grad is not None:
                    comp = name.split(".")[0]
                    ep_grad[comp] += p.grad.data.norm(2).item() ** 2

            nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        ep_loss += info["total"]
        ep_conf_acc += info["conf_acc"]
        for i, v in enumerate(piter):
            ep_iter_snr[i] += v
        n_batches += 1

    # epoch averages
    ep_loss /= n_batches
    ep_conf_acc /= n_batches
    ep_iter_snr = [v / n_batches for v in ep_iter_snr]
    updates_per_epoch = max(1, n_batches // config.grad_accum_steps)
    for comp in ep_grad:
        ep_grad[comp] = math.sqrt(ep_grad[comp] / updates_per_epoch)

    # validation
    val_snri = quick_val(model, val_loader, config)

    # record
    history["train_loss"].append(ep_loss)
    history["val_sisnri"].append(val_snri)
    history["conf_acc"].append(ep_conf_acc)
    for i in range(config.max_iterations):
        history["per_iter_snr"][i].append(ep_iter_snr[i])
    for comp, val in ep_grad.items():
        history["grad_norms"][comp].append(val)

    # save best
    if val_snri > best_val_snri:
        best_val_snri = val_snri
        torch.save(model.state_dict(), f"{config.save_dir}/dme_best.pth")

    # periodic checkpoint (overwrites previous)
    torch.save({"epoch": epoch, "model": model.state_dict(),
                 "opt": optimizer.state_dict(), "history": history},
                f"{config.save_dir}/dme_ckpt_latest.pth")

    elapsed = time.time() - t0
    print(f"Ep {epoch:3d} | loss {ep_loss:.4f} | val SI-SNRi {val_snri:.2f} dB | "
          f"conf_acc {ep_conf_acc:.2%} | lr {lr:.2e} | {elapsed/60:.0f}m")

    torch.cuda.empty_cache()

total_time = time.time() - t0
print(f"\nTraining complete! {total_time/3600:.1f} hours | best val SI-SNRi = {best_val_snri:.2f} dB")

# load best model
model.load_state_dict(torch.load(f"{config.save_dir}/dme_best.pth", map_location=device, weights_only=True))
print("Loaded best model checkpoint")


In [ ]:
# ── Cell 5.3: Training Progress Graphs 5-7 ──
epochs_x = list(range(1, len(history["train_loss"]) + 1))

# ---- Graph 5: Training loss ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_x, history["train_loss"], color=PALETTE[0], linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Graph 5 — Training Loss")
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph05_train_loss.png", dpi=150)
plt.show()

# ---- Graph 6: Validation SI-SNRi ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_x, history["val_sisnri"], color=PALETTE[1], linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("SI-SNRi (dB)"); ax.set_title("Graph 6 — Validation SI-SNRi")
ax.axhline(y=best_val_snri, ls="--", color="grey", alpha=0.6, label=f"best = {best_val_snri:.2f}")
ax.legend()
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph06_val_sisnri.png", dpi=150)
plt.show()

# ---- Graph 7: Confidence head accuracy ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_x, [a * 100 for a in history["conf_acc"]], color=PALETTE[2], linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)"); ax.set_title("Graph 7 — Confidence Head Accuracy")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph07_conf_accuracy.png", dpi=150)
plt.show()
print("Training graphs saved!")


---
## 📊 Section 6: Evaluation on Test Set


In [ ]:
# ── Cell 6.1: Full Evaluation Function ──

@torch.no_grad()
def full_evaluation(model, loader, cfg):
    model.eval()

    res = {
        "si_snri": [], "sdri": [], "pesq": [], "stoi": [],
        "per_iter_sisnri": [[] for _ in range(cfg.max_iterations)],
        "conf_per_iter": [[] for _ in range(cfg.max_iterations)],
        "conf_labels": [], "conf_scores": [],
        "extraction_order": [],
        "durations": [], "input_snrs": [],
    }

    for mix, srcs in tqdm(loader, desc="Evaluating"):
        mix  = mix.to(device)
        srcs_d = [s.to(device) for s in srcs]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            ests, confs = model(mix)
        
        ests = [e.float() for e in ests]
        
        K, N, B = len(ests), len(srcs_d), mix.shape[0]

        for b in range(B):
            mix_np  = mix[b, 0].cpu().numpy()
            src_nps = [s[b, 0].cpu().numpy() for s in srcs_d]
            est_nps = [e[b, 0].cpu().numpy() for e in ests]

            res["durations"].append(len(mix_np) / cfg.sample_rate)
            res["input_snrs"].append(np.mean([si_snr_np(mix_np, s) for s in src_nps]))

            # Hungarian matching
            cost = np.zeros((min(K, N), N))
            for i in range(min(K, N)):
                for j in range(N):
                    cost[i, j] = -si_snr_np(est_nps[i], src_nps[j])
            ri, ci = linear_sum_assignment(cost)

            for i_e, j_t in zip(ri, ci):
                snr_e = si_snr_np(est_nps[i_e], src_nps[j_t])
                snr_m = si_snr_np(mix_np, src_nps[j_t])
                res["si_snri"].append(snr_e - snr_m)

                sdr_e = sdr_np(est_nps[i_e], src_nps[j_t])
                sdr_m = sdr_np(mix_np, src_nps[j_t])
                res["sdri"].append(sdr_e - sdr_m)

                res["per_iter_sisnri"][i_e].append(snr_e - snr_m)

                try:
                    res["pesq"].append(pesq_fn(cfg.sample_rate, src_nps[j_t], est_nps[i_e], "wb"))
                except Exception:
                    pass
                try:
                    res["stoi"].append(stoi_fn(src_nps[j_t], est_nps[i_e], cfg.sample_rate, extended=False))
                except Exception:
                    pass

            # confidence analysis
            for i_c in range(K):
                cv = confs[i_c][b].item()
                res["conf_per_iter"][i_c].append(cv)
                lab = 1 if i_c < N else 0
                res["conf_labels"].append(lab)
                res["conf_scores"].append(cv)

            # extraction order (speaker energy rank)
            energies = [np.sum(s**2) for s in src_nps]
            rank_map = {idx: rank for rank, idx in enumerate(np.argsort(energies)[::-1])}
            order = [-1] * K
            for i_e2, j_t2 in zip(ri, ci):
                order[i_e2] = rank_map[j_t2]
            res["extraction_order"].append(order)

    return res

print("Evaluation function defined")


In [ ]:
# ── Cell 6.2: Run Evaluation ──
print("Running full evaluation on test set (1000 samples) ...")
eval_results = full_evaluation(model, test_loader, config)

mean_sisnri = np.mean(eval_results["si_snri"])
mean_sdri   = np.mean(eval_results["sdri"])
mean_pesq   = np.mean(eval_results["pesq"]) if eval_results["pesq"] else float("nan")
mean_stoi   = np.mean(eval_results["stoi"]) if eval_results["stoi"] else float("nan")

print(f"\n{'='*50}")
print(f"  SI-SNRi : {mean_sisnri:.2f} dB")
print(f"  SDRi    : {mean_sdri:.2f} dB")
print(f"  PESQ    : {mean_pesq:.3f}")
print(f"  STOI    : {mean_stoi:.3f}")
print(f"{'='*50}")


In [ ]:
# ── Cell 6.3: Results Table ──
from IPython.display import Markdown, display

table = f"""
| Model | SI-SNRi (dB) | SDRi (dB) | PESQ | STOI |
|:------|:------------|:---------|:-----|:-----|
| SVoice (literature, N=5) | ~8.5 | — | — | — |
| MOD4 (literature, N=5)   | — | ~13.0 | — | — |
| **DME (ours, {config.num_epochs} epochs)** | **{mean_sisnri:.2f}** | **{mean_sdri:.2f}** | **{mean_pesq:.3f}** | **{mean_stoi:.3f}** |
"""

display(Markdown(table))


---
## 📈 Section 7: Visualization & Analysis Graphs


In [ ]:
# ── Cell 7.1: Graphs 8-10 — Separation Quality ──

# ---- Graph 8: SI-SNRi comparison ----
models  = ["SVoice\n(lit.)", "Conv-TasNet\n(est.)", "MOD4\n(lit. SDRi)", f"DME\n(ours)"]
values  = [8.5, 7.0, 13.0, mean_sisnri]
colours = [PALETTE[4], PALETTE[5], PALETTE[6], PALETTE[1]]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, values, color=colours, edgecolor="white", width=0.55)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{v:.1f}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("dB"); ax.set_title("Graph 8 — SI-SNRi / SDRi Comparison (5 speakers)")
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph08_sisnri_comparison.png", dpi=150)
plt.show()

# ---- Graph 9: PESQ ----
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["DME (ours)"], [mean_pesq], color=PALETTE[1], edgecolor="white", width=0.4)
ax.text(0, mean_pesq + 0.05, f"{mean_pesq:.3f}", ha="center", fontsize=13, fontweight="bold")
ax.set_ylabel("PESQ"); ax.set_title("Graph 9 — PESQ Score"); ax.set_ylim(0, 4.5)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph09_pesq.png", dpi=150)
plt.show()

# ---- Graph 10: STOI ----
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["DME (ours)"], [mean_stoi], color=PALETTE[2], edgecolor="white", width=0.4)
ax.text(0, mean_stoi + 0.02, f"{mean_stoi:.3f}", ha="center", fontsize=13, fontweight="bold")
ax.set_ylabel("STOI"); ax.set_title("Graph 10 — STOI Score"); ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph10_stoi.png", dpi=150)
plt.show()
print("Separation quality graphs saved!")


In [ ]:
# ── Cell 7.2: Graphs 11-12 — Confidence Head Analysis ──

# ---- Graph 11: Confidence histogram per iteration ----
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
for i, ax in enumerate(axes.flat):
    if i < config.max_iterations and eval_results["conf_per_iter"][i]:
        data = eval_results["conf_per_iter"][i]
        colour = PALETTE[1] if i < config.num_speakers else PALETTE[4]
        label  = f"Iter {i+1}" + (" (empty)" if i >= config.num_speakers else "")
        ax.hist(data, bins=30, color=colour, alpha=0.75, edgecolor="white")
        ax.axvline(0.5, ls="--", color="grey")
        ax.set_title(label)
    else:
        ax.set_visible(False)
fig.suptitle("Graph 11 — Confidence Scores per Iteration", fontsize=14)
fig.supxlabel("Confidence"); fig.supylabel("Count")
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph11_conf_histogram.png", dpi=150)
plt.show()

# ---- Graph 12: ROC curve ----
from sklearn.metrics import roc_curve, auc as sk_auc
labels = np.array(eval_results["conf_labels"])
scores = np.array(eval_results["conf_scores"])
if len(np.unique(labels)) > 1:
    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = sk_auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, color=PALETTE[0], lw=2, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0,1],[0,1], ls="--", color="grey")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("Graph 12 — Confidence Head ROC Curve"); ax.legend()
    plt.tight_layout()
    plt.savefig(f"{config.save_dir}/figures/graph12_roc.png", dpi=150)
    plt.show()
else:
    print("Skipping ROC — only one class in labels")
print("Confidence analysis graphs saved!")


In [ ]:
# ── Cell 7.3: Graph 13 — Waveform & Spectrogram Visualisation ──

spec_tf = torchaudio.transforms.Spectrogram(n_fft=512, hop_length=128, power=2.0)
model.eval()

n_samples = 3
fig, axes = plt.subplots(n_samples * 6, 2, figsize=(16, n_samples * 18), gridspec_kw={"width_ratios": [1, 1]})

sample_idx = 0
for s_i in range(n_samples):
    mix, srcs = test_ds[s_i]
    mix_dev = mix.unsqueeze(0).to(device)
    with torch.no_grad():
        ests, _ = model(mix_dev)

    mix_np = mix[0].cpu().numpy()
    L = len(mix_np)
    t_ax = np.arange(L) / config.sample_rate

    # row 0: mixture
    r = s_i * 6
    axes[r, 0].plot(t_ax, mix_np, color=PALETTE[0], linewidth=0.3)
    axes[r, 0].set_title(f"Sample {s_i+1} — Mixture (waveform)")
    spec_db = 10 * torch.log10(spec_tf(mix).clamp(min=1e-10))[0]
    axes[r, 1].imshow(spec_db.numpy(), origin="lower", aspect="auto", cmap="magma",
                      extent=[0, L/config.sample_rate, 0, config.sample_rate/2])
    axes[r, 1].set_title("Mixture (spectrogram)")

    # rows 1-5: ground truth vs estimate
    for sp in range(min(5, len(ests))):
        r2 = r + 1 + sp
        gt_np = srcs[sp][0].numpy()
        est_np = ests[sp][0, 0].cpu().numpy()[:L]
        # waveform overlay
        axes[r2, 0].plot(t_ax, gt_np, color=PALETTE[1], linewidth=0.3, alpha=0.7, label="GT")
        axes[r2, 0].plot(t_ax, est_np, color=PALETTE[3], linewidth=0.3, alpha=0.7, label="Est")
        axes[r2, 0].set_title(f"Speaker {sp+1}")
        if sp == 0:
            axes[r2, 0].legend(fontsize=7)
        # spectrogram of estimate
        est_spec = 10 * torch.log10(spec_tf(torch.tensor(est_np).unsqueeze(0)).clamp(min=1e-10))[0]
        axes[r2, 1].imshow(est_spec.numpy(), origin="lower", aspect="auto", cmap="magma",
                           extent=[0, L/config.sample_rate, 0, config.sample_rate/2])
        axes[r2, 1].set_title(f"Speaker {sp+1} estimate (spectrogram)")

fig.suptitle("Graph 13 — Waveform & Spectrogram Visualisation", fontsize=16, y=1.0)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph13_waveform_spectrogram.png", dpi=120, bbox_inches="tight")
plt.show()
print("Waveform / spectrogram graph saved!")


In [ ]:
# ── Cell 7.4: Graphs 14-16 — Error Analysis ──

# ---- Graph 14: SI-SNRi vs duration ----
fig, ax = plt.subplots(figsize=(8, 5))
# each sample has 5 matched speakers → group by sample
n_test = len(eval_results["durations"])
sisnri_per_sample = []
for i in range(n_test):
    vals = eval_results["si_snri"][i*5:(i+1)*5] if (i+1)*5 <= len(eval_results["si_snri"]) else []
    sisnri_per_sample.append(np.mean(vals) if vals else 0)
ax.scatter(eval_results["durations"][:len(sisnri_per_sample)], sisnri_per_sample,
           alpha=0.4, s=10, color=PALETTE[0])
ax.set_xlabel("Duration (s)"); ax.set_ylabel("Mean SI-SNRi (dB)")
ax.set_title("Graph 14 — SI-SNRi vs Audio Duration")
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph14_sisnri_vs_duration.png", dpi=150)
plt.show()

# ---- Graph 15: SI-SNRi vs input SNR ----
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(eval_results["input_snrs"][:len(sisnri_per_sample)], sisnri_per_sample,
           alpha=0.4, s=10, color=PALETTE[3])
ax.set_xlabel("Input SNR (dB)"); ax.set_ylabel("Mean SI-SNRi (dB)")
ax.set_title("Graph 15 — SI-SNRi vs Input SNR")
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph15_sisnri_vs_inputsnr.png", dpi=150)
plt.show()

# ---- Graph 16: Extraction order heatmap ----
# rows = speaker energy rank (0=loudest), cols = extraction iteration
heat = np.zeros((config.num_speakers, config.max_iterations))
for order in eval_results["extraction_order"]:
    for col, rank in enumerate(order):
        if 0 <= rank < config.num_speakers and col < config.max_iterations:
            heat[rank, col] += 1

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax,
            xticklabels=[f"Iter {i+1}" for i in range(config.max_iterations)],
            yticklabels=[f"Rank {i+1}\n(loud)" if i==0 else f"Rank {i+1}" for i in range(config.num_speakers)])
ax.set_xlabel("Extraction Iteration"); ax.set_ylabel("Speaker Energy Rank")
ax.set_title("Graph 16 — Extraction Order vs Speaker Energy Rank")
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph16_extraction_order.png", dpi=150)
plt.show()
print("Error analysis graphs saved!")


In [ ]:
# ── Cell 7.5: Graphs 17-18 — Training Dynamics ──

# ---- Graph 17: Per-iteration SI-SNR over training ----
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(config.num_speakers):       # only plot the 5 real-speaker iterations
    if history["per_iter_snr"][i]:
        ax.plot(epochs_x, history["per_iter_snr"][i], label=f"Iter {i+1}",
                color=PALETTE[i], linewidth=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("SI-SNR (dB)")
ax.set_title("Graph 17 — Per-Iteration SI-SNR During Training")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph17_per_iter_snr.png", dpi=150)
plt.show()

# ---- Graph 18: Gradient magnitude per component ----
fig, ax = plt.subplots(figsize=(10, 5))
for i, (comp, vals) in enumerate(history["grad_norms"].items()):
    if vals:
        ax.plot(range(1, len(vals)+1), vals, label=comp, color=PALETTE[i % len(PALETTE)], linewidth=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Gradient L2 Norm")
ax.set_title("Graph 18 — Gradient Magnitude per Component")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph18_gradient_norms.png", dpi=150)
plt.show()
print("Training dynamics graphs saved!")


In [ ]:
# ── Cell 7.6: Graph 19 — Inference Time ──

model.eval()
times_per_sample = []
times_per_iter = [[] for _ in range(config.max_iterations)]

for idx in range(min(50, len(test_ds))):
    mix, _ = test_ds[idx]
    mix_dev = mix.unsqueeze(0).to(device)

    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        ests, confs = model(mix_dev)
    torch.cuda.synchronize()
    total_t = time.time() - t0
    times_per_sample.append(total_t)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(["Total\nInference"], [np.mean(times_per_sample) * 1000],
       color=PALETTE[1], edgecolor="white", width=0.3)
ax.text(0, np.mean(times_per_sample)*1000 + 2,
        f"{np.mean(times_per_sample)*1000:.1f} ms", ha="center", fontsize=13, fontweight="bold")
ax.set_ylabel("Time (ms)")
ax.set_title("Graph 19 — Average Inference Time per Sample")
plt.tight_layout()
plt.savefig(f"{config.save_dir}/figures/graph19_inference_time.png", dpi=150)
plt.show()
print(f"Mean inference time: {np.mean(times_per_sample)*1000:.1f} ms / sample")


---
## 🔊 Section 8: Audio Demos


In [ ]:
# ── Cell 8.1: Audio Playback ──
from IPython.display import Audio, display as ipy_display, HTML

model.eval()

for demo_i in range(3):
    print(f"\n{'='*60}")
    print(f"  Demo {demo_i + 1}")
    print(f"{'='*60}")

    mix, srcs = test_ds[demo_i]
    mix_dev = mix.unsqueeze(0).to(device)

    with torch.no_grad():
        ests, confs = model(mix_dev)

    # Mixture
    ipy_display(HTML(f"<b>Mixture</b>"))
    ipy_display(Audio(mix[0].numpy(), rate=config.sample_rate))

    # Ground truth sources
    for sp in range(5):
        ipy_display(HTML(f"<b>Ground Truth — Speaker {sp+1}</b>"))
        ipy_display(Audio(srcs[sp][0].numpy(), rate=config.sample_rate))

    # DME estimates
    for sp in range(min(5, len(ests))):
        est_np = ests[sp][0, 0].cpu().numpy()[:mix.shape[-1]]
        conf_v = confs[sp][0].item()
        ipy_display(HTML(f"<b>DME Estimate — Speaker {sp+1}  (conf={conf_v:.2f})</b>"))
        ipy_display(Audio(est_np, rate=config.sample_rate))

print("\nAudio demos complete!")


---
## 💾 Section 9: Save Results & Model


In [ ]:
# ── Cell 9.1: Save Everything ──

# Save evaluation results
results_df = pd.DataFrame({
    "si_snri": eval_results["si_snri"],
    "sdri":    eval_results["sdri"],
})
results_df.to_csv(f"{config.save_dir}/results.csv", index=False)

# Save training history
import json as json_lib
with open(f"{config.save_dir}/training_history.json", "w") as f:
    serializable = {k: v for k, v in history.items() if k != "grad_norms"}
    serializable["grad_norms"] = dict(history["grad_norms"])
    json_lib.dump(serializable, f)

# Summary
print("=" * 60)
print("  EXPERIMENT SUMMARY")
print("=" * 60)
print(f"  Model           : Deflationary Mamba Extractor (DME)")
print(f"  Dataset          : Libri5Mix (5-speaker)")
print(f"  Epochs           : {config.num_epochs}")
print(f"  Best val SI-SNRi : {best_val_snri:.2f} dB")
print(f"  Test SI-SNRi     : {mean_sisnri:.2f} dB")
print(f"  Test SDRi        : {mean_sdri:.2f} dB")
print(f"  Test PESQ        : {mean_pesq:.3f}")
print(f"  Test STOI        : {mean_stoi:.3f}")
print(f"  Trainable params : {trainable:,}")
print("=" * 60)
print(f"\nSaved to: {config.save_dir}/")
print("  - dme_best.pth          (model weights)")
print("  - results.csv           (per-sample metrics)")
print("  - training_history.json (loss / metric curves)")
print("  - figures/              (19 graphs)")
